# Text Encoder Training — Multi-Modal Disease Detection

**Self-contained Kaggle notebook** that trains a **BioBERT** text encoder on synthetic
clinical symptom descriptions mapped to the 25-class disease label space.

**No external datasets needed** — symptom data is generated synthetically.

**Outputs:**
- `text_encoder.pth`
- `text_head.pth`
- `tokenizer/` directory
- `text_config.json`

In [ ]:
!pip install -q transformers

In [ ]:
import os
import json
import random
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
CONFIG = {
    # Model
    'model_name': 'dmis-lab/biobert-v1.1',
    'hidden_dim': 768,
    'projection_dim': 512,
    'max_length': 128,
    'num_classes': 25,
    'dropout': 0.3,

    # Training
    'batch_size': 32,
    'epochs_frozen': 3,
    'epochs_unfrozen': 10,
    'lr': 2e-5,
    'weight_decay': 0.01,
    'warmup_steps': 100,
    'focal_alpha': 0.75,
    'focal_gamma': 2.0,

    # Data
    'samples_per_disease': 200,
    'val_split': 0.2,
    'seed': SEED,

    # Labels
    'all_labels': [
        'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
        'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
        'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
        'Pleural_Thickening', 'Hernia',
        'Melanoma', 'Melanocytic_Nevi', 'Basal_Cell_Carcinoma',
        'Actinic_Keratosis', 'Benign_Keratosis', 'Dermatofibroma',
        'Vascular_Lesion',
        'Glioma', 'Meningioma', 'Pituitary_Tumor', 'No_Tumor',
    ],

    'output_dir': '/kaggle/working',
}

label_to_idx = {label: i for i, label in enumerate(CONFIG['all_labels'])}
print(f'Classes: {CONFIG["num_classes"]}')

## 2. Synthetic Symptom Generator

In [ ]:
# ── Disease → Symptom Templates ──────────────────────────────────────────

DISEASE_SYMPTOM_MAP = {
    # ==================== CHEST (0-13) ====================
    'Atelectasis': {
        'symptoms': ['reduced breath sounds on one side', 'mild dyspnea', 'low-grade fever',
                      'cough', 'tachypnea', 'chest discomfort'],
        'contexts': ['post-operative patient', 'prolonged bed rest', 'recent intubation'],
        'severity_range': (2, 6),
    },
    'Cardiomegaly': {
        'symptoms': ['progressive dyspnea on exertion', 'orthopnea', 'bilateral lower extremity edema',
                      'fatigue', 'paroxysmal nocturnal dyspnea', 'palpitations', 'jugular venous distention'],
        'contexts': ['known history of hypertension', 'prior myocardial infarction',
                      'valvular heart disease', 'chronic alcohol use'],
        'severity_range': (4, 9),
    },
    'Effusion': {
        'symptoms': ['dyspnea that worsens when lying flat', 'pleuritic chest pain',
                      'reduced breath sounds at the base', 'dullness to percussion', 'dry cough'],
        'contexts': ['known malignancy', 'recent pneumonia', 'heart failure history', 'liver cirrhosis'],
        'severity_range': (3, 7),
    },
    'Infiltration': {
        'symptoms': ['productive cough', 'mild fever', 'fatigue', 'decreased appetite', 'mild chest discomfort'],
        'contexts': ['community-acquired infection', 'immunocompromised state'],
        'severity_range': (2, 5),
    },
    'Mass': {
        'symptoms': ['persistent cough lasting more than 3 weeks', 'unintentional weight loss',
                      'hemoptysis', 'chest pain', 'recurrent pneumonia in the same lobe'],
        'contexts': ['long history of smoking', 'occupational asbestos exposure', 'family history of lung cancer'],
        'severity_range': (4, 8),
    },
    'Nodule': {
        'symptoms': ['incidental finding on chest X-ray', 'no specific symptoms', 'mild persistent cough'],
        'contexts': ['routine health screening', 'smoker over 50 years old', 'history of granulomatous disease'],
        'severity_range': (1, 3),
    },
    'Pneumonia': {
        'symptoms': ['high fever with chills and rigors', 'productive cough with purulent sputum',
                      'severe dyspnea', 'pleuritic chest pain', 'tachycardia', 'malaise and body aches'],
        'contexts': ['recent upper respiratory infection', 'elderly patient in nursing home',
                      'immunosuppressed individual'],
        'severity_range': (5, 9),
    },
    'Pneumothorax': {
        'symptoms': ['sudden onset sharp chest pain', 'acute dyspnea', 'reduced breath sounds on affected side',
                      'hypoxia', 'tachycardia'],
        'contexts': ['tall thin young male', 'recent thoracic procedure', 'underlying COPD', 'mechanical ventilation'],
        'severity_range': (6, 10),
    },
    'Consolidation': {
        'symptoms': ['productive cough', 'fever', 'bronchial breath sounds', 'increased tactile fremitus', 'malaise'],
        'contexts': ['community-acquired pneumonia', 'aspiration event'],
        'severity_range': (4, 7),
    },
    'Edema': {
        'symptoms': ['severe dyspnea at rest', 'orthopnea requiring multiple pillows', 'pink frothy sputum',
                      'bilateral crackles on auscultation', 'anxiety and restlessness'],
        'contexts': ['acute heart failure exacerbation', 'fluid overload', 'renal failure'],
        'severity_range': (7, 10),
    },
    'Emphysema': {
        'symptoms': ['chronic progressive dyspnea', 'pursed lip breathing', 'barrel chest appearance',
                      'prolonged expiratory phase', 'decreased breath sounds diffusely', 'exercise intolerance'],
        'contexts': ['long-term smoker', 'alpha-1 antitrypsin deficiency'],
        'severity_range': (4, 8),
    },
    'Fibrosis': {
        'symptoms': ['progressive exertional dyspnea', 'dry non-productive cough',
                      'fine bibasilar crackles (velcro-like)', 'digital clubbing', 'fatigue'],
        'contexts': ['idiopathic pulmonary fibrosis', 'occupational dust exposure', 'connective tissue disease'],
        'severity_range': (4, 8),
    },
    'Pleural_Thickening': {
        'symptoms': ['mild chronic chest pain', 'decreased breath sounds at the base', 'mild dyspnea on exertion'],
        'contexts': ['asbestos exposure history', 'previous pleural infection', 'prior hemothorax'],
        'severity_range': (2, 5),
    },
    'Hernia': {
        'symptoms': ['epigastric fullness after meals', 'shortness of breath when supine',
                      'bowel sounds heard in the chest', 'nausea and vomiting'],
        'contexts': ['previous abdominal surgery', 'congenital diaphragmatic defect',
                      'traumatic injury to the diaphragm'],
        'severity_range': (3, 7),
    },
    # ==================== SKIN (14-20) ====================
    'Melanoma': {
        'symptoms': ['asymmetric mole with irregular borders', 'mole with multiple colors',
                      'diameter greater than 6 mm', 'recent change in size shape or color',
                      'itching or bleeding from a mole', 'new pigmented lesion that looks different'],
        'contexts': ['fair-skinned individual with history of sunburns', 'family history of melanoma',
                      'large number of atypical nevi', 'prior melanoma diagnosis'],
        'severity_range': (6, 10),
    },
    'Melanocytic_Nevi': {
        'symptoms': ['small round symmetric mole', 'uniform brown color', 'well-defined borders',
                      'diameter less than 6 mm', 'stable appearance over time'],
        'contexts': ['routine skin examination', 'cosmetic concern'],
        'severity_range': (1, 2),
    },
    'Basal_Cell_Carcinoma': {
        'symptoms': ['pearly or waxy bump on the skin', 'flat flesh-colored or brown scar-like lesion',
                      'bleeding or scabbing sore that heals and returns', 'visible telangiectatic vessels'],
        'contexts': ['sun-exposed area face neck ears', 'chronic sun exposure history', 'fair skin type'],
        'severity_range': (3, 6),
    },
    'Actinic_Keratosis': {
        'symptoms': ['rough scaly patch on the skin', 'pink or red raised area',
                      'tenderness when touched', 'may itch intermittently'],
        'contexts': ['chronic sun exposure', 'age over 50', 'light skin complexion'],
        'severity_range': (2, 5),
    },
    'Benign_Keratosis': {
        'symptoms': ['waxy stuck-on looking growth', 'well-circumscribed with regular borders',
                      'brown or tan colored', 'slight itching when irritated'],
        'contexts': ['age-related skin change', 'trunk or back location'],
        'severity_range': (1, 3),
    },
    'Dermatofibroma': {
        'symptoms': ['firm hard nodule under the skin', 'dimple sign when pinched',
                      'brownish or reddish color', 'slightly tender to touch'],
        'contexts': ['previous insect bite at the site', 'lower extremity location'],
        'severity_range': (1, 3),
    },
    'Vascular_Lesion': {
        'symptoms': ['red or purple patch on the skin', 'lesion blanches with pressure',
                      'clearly visible blood vessels', 'no pain or itching'],
        'contexts': ['present since birth or early childhood', 'cosmetic concern'],
        'severity_range': (1, 3),
    },
    # ==================== BRAIN (21-24) ====================
    'Glioma': {
        'symptoms': ['progressive headaches worse in the morning', 'new onset seizures',
                      'focal neurological deficits', 'personality or cognitive changes',
                      'nausea and vomiting from raised intracranial pressure', 'visual disturbances',
                      'speech difficulties'],
        'contexts': ['previously healthy adult', 'no prior history of seizures'],
        'severity_range': (6, 10),
    },
    'Meningioma': {
        'symptoms': ['gradually worsening headaches', 'seizures', 'visual changes or visual field loss',
                      'weakness in the extremities', 'personality changes', 'hearing loss on one side'],
        'contexts': ['middle-aged female', 'history of cranial radiation', 'incidental finding on imaging'],
        'severity_range': (3, 7),
    },
    'Pituitary_Tumor': {
        'symptoms': ['bitemporal hemianopsia', 'headaches', 'amenorrhea or irregular periods',
                      'galactorrhea', 'fatigue and weight gain', 'decreased libido', 'visual acuity changes'],
        'contexts': ['young adult female', 'elevated prolactin levels', 'Cushing syndrome features'],
        'severity_range': (3, 7),
    },
    'No_Tumor': {
        'symptoms': ['tension-type headache', 'migraine with aura', 'non-specific dizziness',
                      'anxiety-related symptoms', 'no focal neurological deficits'],
        'contexts': ['MRI ordered to rule out structural pathology', 'normal neurological examination',
                      'reassurance visit'],
        'severity_range': (1, 3),
    },
}

# Co-occurring disease pairs
CO_OCCURRING_PAIRS = [
    ('Pneumonia', 'Effusion'), ('Pneumonia', 'Consolidation'),
    ('Cardiomegaly', 'Edema'), ('Cardiomegaly', 'Effusion'),
    ('Emphysema', 'Pneumothorax'), ('Infiltration', 'Atelectasis'),
    ('Mass', 'Effusion'), ('Fibrosis', 'Pleural_Thickening'),
]

AGES = list(range(18, 90))
GENDERS = ['male', 'female']
DURATIONS = [
    'for the past few days', 'for about a week', 'for two weeks',
    'for the past month', 'over the last several months',
    'for approximately three months', 'gradually over the past year',
]
SEVERITY_WORDS = {
    (1, 3): ['mild', 'slight', 'minimal'],
    (4, 6): ['moderate', 'noticeable', 'persistent'],
    (7, 8): ['severe', 'significant', 'debilitating'],
    (9, 10): ['critical', 'life-threatening', 'acute'],
}
TEMPLATES = [
    'A {age}-year-old {gender} presents with {symptoms}. {context}. '
    'Symptoms have been present {duration}. Severity assessed as {severity}.',
    'Patient is a {age}-year-old {gender} complaining of {symptoms} '
    '{duration}. Clinical context: {context}. Current severity: {severity}.',
    '{age} y/o {gender}. Chief complaint: {symptoms}. Duration: {duration}. '
    'Background: {context}. Severity: {severity}/10.',
    'History: {age}-year-old {gender} reports {symptoms} {duration}. '
    'Relevant context includes {context}. '
    'On a scale of 1-10 the patient rates severity as {severity}.',
    'Clinical presentation: {gender}, age {age}. {symptoms}. '
    'Onset: {duration}. Additional context: {context}. '
    'Assessed severity: {severity}/10.',
]


def _severity_word(score, rng):
    for (lo, hi), words in SEVERITY_WORDS.items():
        if lo <= score <= hi:
            return rng.choice(words)
    return 'moderate'


def generate_symptom_text(disease, rng):
    info = DISEASE_SYMPTOM_MAP[disease]
    age = rng.choice(AGES)
    gender = rng.choice(GENDERS)
    duration = rng.choice(DURATIONS)
    severity_score = rng.randint(*info['severity_range'])
    severity = f"{_severity_word(severity_score, rng)} ({severity_score}/10)"
    n_sym = rng.randint(2, min(5, len(info['symptoms'])))
    symptoms = rng.sample(info['symptoms'], n_sym)
    symptom_str = ', '.join(symptoms[:-1]) + f', and {symptoms[-1]}'
    context = rng.choice(info['contexts'])
    template = rng.choice(TEMPLATES)
    return template.format(
        age=age, gender=gender, symptoms=symptom_str,
        context=context, duration=duration, severity=severity
    )


def generate_multilabel_text(diseases, rng):
    all_symptoms = []
    all_contexts = []
    max_severity = 0
    for disease in diseases:
        info = DISEASE_SYMPTOM_MAP[disease]
        n = rng.randint(1, min(3, len(info['symptoms'])))
        all_symptoms.extend(rng.sample(info['symptoms'], n))
        all_contexts.append(rng.choice(info['contexts']))
        max_severity = max(max_severity, rng.randint(*info['severity_range']))
    # Dedup
    seen = set()
    unique = []
    for s in all_symptoms:
        if s not in seen:
            unique.append(s); seen.add(s)
    age = rng.choice(AGES)
    gender = rng.choice(GENDERS)
    duration = rng.choice(DURATIONS)
    severity = f"{_severity_word(max_severity, rng)} ({max_severity}/10)"
    symptom_str = ', '.join(unique[:-1]) + f', and {unique[-1]}'
    context = '; '.join(all_contexts)
    template = rng.choice(TEMPLATES)
    return template.format(
        age=age, gender=gender, symptoms=symptom_str,
        context=context, duration=duration, severity=severity
    )


def generate_dataset(samples_per_disease=200, seed=42):
    rng = random.Random(seed)
    dataset = []
    # Single-label samples
    for disease in DISEASE_SYMPTOM_MAP:
        for _ in range(samples_per_disease):
            text = generate_symptom_text(disease, rng)
            dataset.append((text, [disease]))
    # Multi-label samples
    multi_per_pair = samples_per_disease // 4
    for d1, d2 in CO_OCCURRING_PAIRS:
        for _ in range(multi_per_pair):
            text = generate_multilabel_text([d1, d2], rng)
            dataset.append((text, [d1, d2]))
    rng.shuffle(dataset)
    return dataset


# Generate!
raw_data = generate_dataset(
    samples_per_disease=CONFIG['samples_per_disease'],
    seed=SEED,
)
print(f'Generated {len(raw_data)} symptom samples')
print(f'\nExample:\n{raw_data[0][0]}\n  Labels: {raw_data[0][1]}')

In [ ]:
# ── Convert to multi-hot labels ──────────────────────────────────────────

texts = []
labels_multihot = []

for text, diseases in raw_data:
    texts.append(text)
    vec = np.zeros(CONFIG['num_classes'], dtype=np.float32)
    for d in diseases:
        vec[label_to_idx[d]] = 1.0
    labels_multihot.append(vec)

labels_multihot = np.array(labels_multihot)

# Train/val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels_multihot,
    test_size=CONFIG['val_split'],
    random_state=SEED,
)

print(f'Train: {len(train_texts)}, Val: {len(val_texts)}')

# Class distribution
print(f'\nLabel distribution (train):')
for i, label in enumerate(CONFIG['all_labels']):
    count = int(train_labels[:, i].sum())
    print(f'  {label:25s}: {count}')

## 3. Model Definitions

In [ ]:
# ── Text Encoder ─────────────────────────────────────────────────────────

class TextEncoder(nn.Module):
    def __init__(
        self,
        model_name='dmis-lab/biobert-v1.1',
        hidden_dim=768,
        projection_dim=512,
        dropout=0.3,
        max_length=128,
        freeze_bert=False,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.projection_dim = projection_dim
        self.max_length = max_length
        self.bert = AutoModel.from_pretrained(model_name, output_attentions=True)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.last_attention_weights = None

    def _encode(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
        last_hidden = outputs.last_hidden_state
        attn_last = outputs.attentions[-1]
        attn_avg = attn_last.mean(dim=1)
        self.last_attention_weights = attn_avg.detach()
        return last_hidden, attn_avg

    def forward(self, input_ids, attention_mask):
        last_hidden, _ = self._encode(input_ids, attention_mask)
        cls_token = last_hidden[:, 0, :]
        return self.projection(cls_token)

    def get_token_features(self, input_ids, attention_mask):
        last_hidden, _ = self._encode(input_ids, attention_mask)
        return self.projection(last_hidden)

    def freeze_bert(self):
        for param in self.bert.parameters():
            param.requires_grad = False

    def unfreeze_bert(self, last_n_layers=2):
        layers = self.bert.encoder.layer
        for layer in layers[-last_n_layers:]:
            for param in layer.parameters():
                param.requires_grad = True


class TextClassifier(nn.Module):
    def __init__(self, encoder, num_classes=25, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(encoder.projection_dim, encoder.projection_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(encoder.projection_dim // 2, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        emb = self.encoder(input_ids, attention_mask)
        return self.head(emb)


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        p = torch.sigmoid(inputs)
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss = focal_weight * ce_loss
        return loss.mean() if self.reduction == 'mean' else loss.sum()


print('Models defined.')

In [ ]:
# ── Dataset + DataLoaders ─────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])


class SymptomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float32),
        }


train_dataset = SymptomDataset(train_texts, train_labels, tokenizer, CONFIG['max_length'])
val_dataset = SymptomDataset(val_texts, val_labels, tokenizer, CONFIG['max_length'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                           num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 4. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc='Train', leave=False):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, (all_preds >= 0.5).astype(int), average='macro', zero_division=0)
    return avg_loss, f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc='Val', leave=False):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        with autocast():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, (all_preds >= 0.5).astype(int), average='macro', zero_division=0)

    valid_cols = [
        c for c in range(all_targets.shape[1])
        if all_targets[:, c].sum() > 0 and all_targets[:, c].sum() < len(all_targets)
    ]
    roc_auc = roc_auc_score(
        all_targets[:, valid_cols], all_preds[:, valid_cols], average='macro'
    ) if valid_cols else 0.0

    return avg_loss, f1, roc_auc, all_preds, all_targets

In [ ]:
# ── Build Models ──────────────────────────────────────────────────────────

encoder = TextEncoder(
    model_name=CONFIG['model_name'],
    hidden_dim=CONFIG['hidden_dim'],
    projection_dim=CONFIG['projection_dim'],
    dropout=CONFIG['dropout'],
    max_length=CONFIG['max_length'],
    freeze_bert=True,  # Start frozen
)
text_classifier = TextClassifier(encoder, CONFIG['num_classes'], CONFIG['dropout']).to(DEVICE)

criterion = FocalLoss(alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'])
scaler = GradScaler()

# ── Phase 1: Frozen BERT ─────────────────────────────────────────────────
print('='*60)
print('Phase 1: Training with FROZEN BioBERT')
print('='*60)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, text_classifier.parameters()),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
)
total_steps = len(train_loader) * CONFIG['epochs_frozen']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG['warmup_steps'],
    num_training_steps=total_steps,
)

best_val_f1 = 0.0
history = []

for epoch in range(CONFIG['epochs_frozen']):
    train_loss, train_f1 = train_one_epoch(
        text_classifier, train_loader, criterion, optimizer, scheduler, scaler, DEVICE
    )
    val_loss, val_f1, val_auc, _, _ = validate(
        text_classifier, val_loader, criterion, DEVICE
    )
    history.append({
        'epoch': epoch + 1, 'phase': 'frozen',
        'train_loss': train_loss, 'train_f1': train_f1,
        'val_loss': val_loss, 'val_f1': val_f1, 'val_auc': val_auc,
    })
    print(
        f'Epoch {epoch+1}/{CONFIG["epochs_frozen"]} (frozen) | '
        f'Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | '
        f'Val Loss: {val_loss:.4f} F1: {val_f1:.4f} AUC: {val_auc:.4f}'
    )
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(encoder.state_dict(), os.path.join(CONFIG['output_dir'], 'text_encoder_best.pth'))

# ── Phase 2: Unfreeze last 2 layers ──────────────────────────────────────
print('\n' + '='*60)
print('Phase 2: Training with UNFROZEN BioBERT (last 2 layers)')
print('='*60)

encoder.unfreeze_bert(last_n_layers=2)

optimizer = torch.optim.AdamW([
    {'params': encoder.projection.parameters(), 'lr': CONFIG['lr']},
    {'params': encoder.bert.encoder.layer[-1].parameters(), 'lr': CONFIG['lr'] * 0.1},
    {'params': encoder.bert.encoder.layer[-2].parameters(), 'lr': CONFIG['lr'] * 0.1},
    {'params': text_classifier.head.parameters(), 'lr': CONFIG['lr']},
], weight_decay=CONFIG['weight_decay'])

total_steps = len(train_loader) * CONFIG['epochs_unfrozen']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=min(CONFIG['warmup_steps'], total_steps // 10),
    num_training_steps=total_steps,
)

for epoch in range(CONFIG['epochs_unfrozen']):
    train_loss, train_f1 = train_one_epoch(
        text_classifier, train_loader, criterion, optimizer, scheduler, scaler, DEVICE
    )
    val_loss, val_f1, val_auc, _, _ = validate(
        text_classifier, val_loader, criterion, DEVICE
    )
    history.append({
        'epoch': CONFIG['epochs_frozen'] + epoch + 1, 'phase': 'unfrozen',
        'train_loss': train_loss, 'train_f1': train_f1,
        'val_loss': val_loss, 'val_f1': val_f1, 'val_auc': val_auc,
    })
    print(
        f'Epoch {epoch+1}/{CONFIG["epochs_unfrozen"]} (unfrozen) | '
        f'Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | '
        f'Val Loss: {val_loss:.4f} F1: {val_f1:.4f} AUC: {val_auc:.4f}'
    )
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(encoder.state_dict(), os.path.join(CONFIG['output_dir'], 'text_encoder_best.pth'))

print(f'\nBest validation F1: {best_val_f1:.4f}')

## 5. Evaluation

In [ ]:
# ── Final Evaluation ──────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve

best_path = os.path.join(CONFIG['output_dir'], 'text_encoder_best.pth')
if os.path.exists(best_path):
    encoder.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print('Loaded best encoder weights.')

val_loss, val_f1, val_auc, all_preds, all_targets = validate(
    text_classifier, val_loader, criterion, DEVICE
)

# ── Per-class optimal threshold via precision-recall curve ────────────────
optimal_thresholds = np.full(CONFIG['num_classes'], 0.5)

for i in range(CONFIG['num_classes']):
    pos_count = all_targets[:, i].sum()
    if pos_count < 2:
        continue
    precision, recall, thresholds = precision_recall_curve(
        all_targets[:, i], all_preds[:, i]
    )
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_vals = np.where(
            (precision + recall) > 0,
            2 * precision * recall / (precision + recall),
            0.0,
        )
    best_idx = np.argmax(f1_vals[:-1])
    optimal_thresholds[i] = thresholds[best_idx]

print('Optimal per-class thresholds:')
for i, label in enumerate(CONFIG['all_labels']):
    if all_targets[:, i].sum() > 0:
        print(f'  {label:25s}  threshold={optimal_thresholds[i]:.3f}')

y_bin_fixed = (all_preds >= 0.5).astype(int)
y_bin_opt = (all_preds >= optimal_thresholds[None, :]).astype(int)

f1_fixed = f1_score(all_targets, y_bin_fixed, average='macro', zero_division=0)
f1_opt = f1_score(all_targets, y_bin_opt, average='macro', zero_division=0)

print(f'\nFinal Validation Results:')
print(f'  Loss:                {val_loss:.4f}')
print(f'  ROC-AUC:             {val_auc:.4f}')
print(f'  Macro F1 (τ=0.5):   {f1_fixed:.4f}')
print(f'  Macro F1 (optimal τ): {f1_opt:.4f}')

print(f'\nPer-class F1 scores (optimal thresholds):')
print('-' * 55)
for i, label in enumerate(CONFIG['all_labels']):
    if all_targets[:, i].sum() > 0:
        cls_f1_fixed = f1_score(all_targets[:, i], y_bin_fixed[:, i], zero_division=0)
        cls_f1_opt = f1_score(all_targets[:, i], y_bin_opt[:, i], zero_division=0)
        count = int(all_targets[:, i].sum())
        delta = cls_f1_opt - cls_f1_fixed
        marker = ' ▲' if delta > 0.01 else ''
        print(f'  {label:25s}  F1={cls_f1_opt:.4f}  (was {cls_f1_fixed:.4f}, τ={optimal_thresholds[i]:.3f}, n={count}){marker}')


In [ ]:
# ── Training History Plot ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_x = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
val_f1s = [h['val_f1'] for h in history]
val_aucs = [h['val_auc'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_x, train_losses, label='Train')
axes[0].plot(epochs_x, val_losses, label='Val')
axes[0].axvline(x=CONFIG['epochs_frozen'], color='red', linestyle='--', alpha=0.5, label='Unfreeze')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs_x, val_f1s, 'g-')
axes[1].axvline(x=CONFIG['epochs_frozen'], color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1'); axes[1].set_title('Validation Macro F1')

axes[2].plot(epochs_x, val_aucs, 'm-')
axes[2].axvline(x=CONFIG['epochs_frozen'], color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC'); axes[2].set_title('Validation ROC-AUC')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'text_training_history.png'), dpi=150)
plt.show()

## 6. Save Models

In [ ]:
output_dir = CONFIG['output_dir']

# Save encoder
torch.save(encoder.state_dict(), os.path.join(output_dir, 'text_encoder.pth'))

# Save text classification head
torch.save(text_classifier.head.state_dict(), os.path.join(output_dir, 'text_head.pth'))

# Save tokenizer
tokenizer_dir = os.path.join(output_dir, 'tokenizer')
os.makedirs(tokenizer_dir, exist_ok=True)
tokenizer.save_pretrained(tokenizer_dir)
print(f'Tokenizer saved to {tokenizer_dir}')

# Config JSON
text_config = {
    'model_name': CONFIG['model_name'],
    'hidden_dim': CONFIG['hidden_dim'],
    'projection_dim': CONFIG['projection_dim'],
    'max_length': CONFIG['max_length'],
    'num_classes': CONFIG['num_classes'],
    'dropout': CONFIG['dropout'],
    'all_labels': CONFIG['all_labels'],
    'optimal_thresholds': optimal_thresholds.tolist(),
    'best_val_f1': best_val_f1,
    'best_val_f1_optimal': float(f1_opt),
    'training_samples': len(train_texts),
    'training_history': history,
}

with open(os.path.join(output_dir, 'text_config.json'), 'w') as f:
    json.dump(text_config, f, indent=2)

# List outputs
print('\nSaved files:')
for root, dirs, files in os.walk(output_dir):
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        rel = os.path.relpath(fpath, output_dir)
        print(f'  {rel:50s} {size_mb:.1f} MB')

## 7. GPT Augmentation Prompt

For richer symptom diversity, use this prompt with GPT-4 to generate more training data:

In [ ]:
GPT_PROMPT = """
You are a medical text generation assistant. Given the disease label and
a seed symptom description, generate 5 diverse paraphrased versions of
the clinical presentation.

Requirements:
- Maintain medical accuracy
- Vary the writing style (clinical note, patient self-report, ER triage)
- Randomise demographics (age, gender)
- Keep the core symptoms consistent with the disease
- Include relevant physical exam findings
- Each version should be 2-4 sentences

Disease: {disease}
Seed description: {seed_text}

Generate 5 variations:
"""

# Example usage
example_disease = 'Pneumonia'
example_seed = generate_symptom_text(example_disease, random.Random(0))
print('Example GPT augmentation prompt:')
print('='*60)
print(GPT_PROMPT.format(disease=example_disease, seed_text=example_seed))